<a href="https://colab.research.google.com/github/vcellmike/PatternsFormation/blob/main/Working/2024_08_21_XGBoost_on_ImageJ_Features.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [9]:
import json
import os
import torch
import torch.nn as nn
import torchvision.transforms as transforms
from sklearn.metrics import accuracy_score, classification_report
from transformers import AutoModel, AutoTokenizer, get_scheduler
from torch.utils.data import Dataset, DataLoader, RandomSampler, SequentialSampler
from torch.optim import AdamW
from tqdm.notebook import tqdm, trange
from time import perf_counter
from PIL import Image
import pandas as pd
#from google.colab import drive
from PIL import Image
import pandas as pd
import os
import numpy as np
import matplotlib.pyplot as plt
import pickle

print("All dependencies present.")

All dependencies present.


In [10]:
# set random seeds for repeatability
import numpy as np
import random

def set_seed(seed_val):
    random.seed(seed_val)
    np.random.seed(seed_val)
    torch.manual_seed(seed_val)
    torch.cuda.manual_seed_all(seed_val)

print("Seed set")

Seed set


In [11]:
seed_val = 42
set_seed(seed_val)

In [12]:
file_in = "./data/Images_Classified_np126"

with open(file_in + '.pkl', 'rb') as f:
    feats_df5 = pickle.load(f)

#load in dataframe
pd.set_option('display.max_columns', None)

file_out = file_in + "_s" + str(seed_val) + "_XGB" + ".model"

print(feats_df5.shape)

feats_df5.head()

(27515, 113)


,Ua,Ui,Ga,Gi,Ba,Da,Di,pattern,noise,path,seed,dir,feat_bool,num_spots,Mean,Median,Area_mean,Area_std,X_mean,X_std,Y_mean,Y_std,Perim._mean,Perim._std,BX_mean,BX_std,BY_mean,BY_std,Width_mean,Width_std,Height_mean,Height_std,Major_mean,Major_std,Minor_mean,Minor_std,Angle_mean,Angle_std,Circ._mean,Circ._std,Feret_mean,Feret_std,IntDen_mean,IntDen_std,%Area_mean,%Area_std,RawIntDen_mean,RawIntDen_std,FeretX_mean,FeretX_std,FeretY_mean,FeretY_std,FeretAngle_mean,FeretAngle_std,MinFeret_mean,MinFeret_std,AR_mean,AR_std,Round_mean,Round_std,Solidity_mean,Solidity_std,num_spots_inverted,Mean_inverted,Median_inverted,Area_inverted_mean,Area_inverted_std,X_inverted_mean,X_inverted_std,Y_inverted_mean,Y_inverted_std,Perim._inverted_mean,Perim._inverted_std,BX_inverted_mean,BX_inverted_std,BY_inverted_mean,BY_inverted_std,Width_inverted_mean,Width_inverted_std,Height_inverted_mean,Height_inverted_std,Major_inverted_mean,Major_inverted_std,Minor_inverted_mean,Minor_inverted_std,Angle_inverted_mean,Angle_inverted_std,Circ._inverted_mean,Circ._inverted_std,Feret_inverted_mean,Feret_inverted_std,IntDen_inverted_mean,IntDen_inverted_std,%Area_inverted_mean,%Area_inverted_std,RawIntDen_inverted_mean,RawIntDen_inverted_std,FeretX_inverted_mean,FeretX_inverted_std,FeretY_inverted_mean,FeretY_inverted_std,FeretAngle_inverted_mean,FeretAngle_inverted_std,MinFeret_inverted_mean,MinFeret_inverted_std,AR_inverted_mean,AR_inverted_std,Round_inverted_mean,Round_inverted_std,Solidity_inverted_mean,Solidity_inverted_std,full_path,classifier_pred_class
0,0.025122,0.063822,0.071957,0.106327,-0.113131,0.009186,0.611871,1,2,1.png,1,/content/Images3,1,70,5.489,0,12.300000,2.548669,99.395943,58.506181,97.530229,58.913237,12.282143,1.400962,97.371429,58.453442,95.528571,58.843551,4.071429,0.723512,4.057143,0.753766,4.245571,0.439294,3.664200,0.511781,55.108900,57.413924,0.958686,0.069883,4.926700,0.412294,3136.500000,649.910626,100.0,0.0,3136.500000,649.910626,97.942857,58.572760,96.014286,58.843981,120.129000,30.775307,3.653171,0.564778,1.177686,0.183575,0.866686,0.115368,0.879486,0.067138,1,249.511,255,39139.0,0.0,99.998,0.0,100.022,0.0,811.882,0.0,0.0,0.0,0.0,0.0,200.0,0.0,200.0,0.0,223.351,0.0,223.117,0.0,38.796,0.0,0.746,0.0,282.843,0.0,9980445.0,0.0,100.0,0.0,9980445.0,0.0,0.0,0.0,200.0,0.0,45.0,0.0,200.0,0.0,1.001,0.0,0.999,0.0,0.979,0.0,/content/Images3/1.png,2
1,0.034475,0.059956,0.080798,0.10037,-0.132577,0.008852,0.854084,1,2,3.png,3,/content/Images3,1,63,4.921,0,12.253968,2.569468,101.800571,59.297600,99.451270,58.953924,11.663270,1.365051,99.968254,59.323693,97.619048,58.964458,3.666667,0.534522,3.666667,0.534522,4.237317,0.401713,3.661714,0.572434,56.049556,60.605082,0.999619,0.003000,4.908365,0.491745,3124.761905,655.214389,100.0,0.0,3124.761905,655.214389,100.158730,59.330446,98.079365,59.017431,120.191190,32.406426,3.476190,0.613529,1.185032,0.226780,0.867667,0.127976,0.954905,0.046408,1,250.078,255,39228.0,0.0,99.992,0.0,100.015,0.0,809.255,0.0,0.0,0.0,0.0,0.0,200.0,0.0,200.0,0.0,223.524,0.0,223.451,0.0,72.380,0.0,0.753,0.0,282.843,0.0,10003140.0,0.0,100.0,0.0,10003140.0,0.0,0.0,0.0,0.0,0.0,135.0,0.0,200.0,0.0,1.000,0.0,1.000,0.0,0.981,0.0,/content/Images3/3.png,2
2,0.028367,0.057912,0.073998,0.080784,-0.133789,0.008512,0.534244,1,2,7.png,7,/content/Images3,1,77,4.692,0,9.558442,1.427863,100.557675,58.022388,98.103338,59.633687,10.379961,1.091819,98.870130,58.006796,96.519481,59.576560,3.402597,0.564300,3.194805,0.559196,3.806623,0.403173,3.196818,0.352550,29.220831,46.845042,0.992974,0.032896,4.416805,0.320359,2437.402597,364.105032,100.0,0.0,2437.402597,364.105032,98.961039,57.996292,96.753247,59.526521,133.450130,23.616781,2.961039,0.375951,1.209857,0.220792,0.849429,0.129524,0.946805,0.050966,1,250.308,255,39264.0,0.0,99.987,0.0,100.001,0.0,808.326,0.0,0.0,0.0,0.0,0.0,200.0,0.0,200.0,0.0,223.626,0.0,223.554,0.0,5.766,0.0,0.755,0.0,282.843,0.0,10012320.0,0.0,100.0,0.0,10012320.0,0.0,0.0,0.0,200.0,0.0,45.0,0.0,200.0,0.0,1.000,0.0,1.000,0.0,

In [13]:
#with open("./data/data_old/2024-08-19_feat_array_unscaled.pkl", 'rb') as f:
#  feat_array_classified = pickle.load(f) # deserialize using load()

#with open("./data/data_old/2024-08-18_curr_df_np126.pkl", 'rb') as f:
#  df_classes_classified = pickle.load(f) # deserialize using load()

#feat_array_classified

In [14]:
# Features, 33k images
with open("./data/data_old/2024-08-19_feats_df_narrow_np126.pkl", 'rb') as f:
  feats_df_new = pickle.load(f) # deserialize using load()

feats_df_new

,Ua,Ui,Ga,Gi,Ba,Da,Di,pattern,noise,path,seed,dir,feat_bool,num_spots,Mean,Median,Area_mean,Area_std,X_mean,X_std,Y_mean,Y_std,Perim._mean,Perim._std,BX_mean,BX_std,BY_mean,BY_std,Width_mean,Width_std,Height_mean,Height_std,Major_mean,Major_std,Minor_mean,Minor_std,Angle_mean,Angle_std,Circ._mean,Circ._std,Feret_mean,Feret_std,IntDen_mean,IntDen_std,%Area_mean,%Area_std,RawIntDen_mean,RawIntDen_std,FeretX_mean,FeretX_std,FeretY_mean,FeretY_std,FeretAngle_mean,FeretAngle_std,MinFeret_mean,MinFeret_std,AR_mean,AR_std,Round_mean,Round_std,Solidity_mean,Solidity_std,num_spots_inverted,Mean_inverted,Median_inverted,Area_inverted_mean,Area_inverted_std,X_inverted_mean,X_inverted_std,Y_inverted_mean,Y_inverted_std,Perim._inverted_mean,Perim._inverted_std,BX_inverted_mean,BX_inverted_std,BY_inverted_mean,BY_inverted_std,Width_inverted_mean,Width_inverted_std,Height_inverted_mean,Height_inverted_std,Major_inverted_mean,Major_inverted_std,Minor_inverted_mean,Minor_inverted_std,Angle_inverted_mean,Angle_inverted_std,Circ._inverted_mean,Circ._inverted_std,Feret_inverted_mean,Feret_inverted_std,IntDen_inverted_mean,IntDen_inverted_std,%Area_inverted_mean,%Area_inverted_std,RawIntDen_inverted_mean,RawIntDen_inverted_std,FeretX_inverted_mean,FeretX_inverted_std,FeretY_inverted_mean,FeretY_inverted_std,FeretAngle_inverted_mean,FeretAngle_inverted_std,MinFeret_inverted_mean,MinFeret_inverted_std,AR_inverted_mean,AR_inverted_std,Round_inverted_mean,Round_inverted_std,Solidity_inverted_mean,Solidity_inverted_std
0,0.025122,0.063822,0.071957,0.106327,-0.113131,0.009186,0.611871,1,2,1.png,1,/content/Images3/,1,70,5.489,0,12.300000,2.548669,99.395943,58.506181,97.530229,58.913237,12.282143,1.400962,97.371429,58.453442,95.528571,58.843551,4.071429,0.723512,4.057143,0.753766,4.245571,0.439294,3.664200,0.511781,55.108900,57.413924,0.958686,0.069883,4.926700,0.412294,3.136500e+03,649.910626,100.0,0.0,3.136500e+03,649.910626,97.942857,58.572760,96.014286,58.843981,120.129000,30.775307,3.653171,0.564778,1.177686,0.183575,0.866686,0.115368,0.879486,0.067138,1,249.511,255,39139.0,0.0,99.998,0.0,100.022,0.0,811.882,0.0,0.0,0.0,0.0,0.0,200.0,0.0,200.0,0.0,223.351,0.0,223.117,0.0,38.796,0.0,0.746,0.0,282.843,0.0,9980445.0,0.0,100.0,0.0,9980445.0,0.0,0.0,0.0,200.0,0.0,45.000,0.0,200.0,0.0,1.001,0.0,0.999,0.0,0.979,0.0
1,0.034475,0.059956,0.080798,0.10037,-0.132577,0.008852,0.854084,1,2,3.png,3,/content/Images3/,1,63,4.921,0,12.253968,2.569468,101.800571,59.297600,99.451270,58.953924,11.663270,1.365051,99.968254,59.323693,97.619048,58.964458,3.666667,0.534522,3.666667,0.534522,4.237317,0.401713,3.661714,0.572434,56.049556,60.605082,0.999619,0.003000,4.908365,0.491745,3.124762e+03,655.214389,100.0,0.0,3.124762e+03,655.214389,100.158730,59.330446,98.079365,59.017431,120.191190,32.406426,3.476190,0.613529,1.185032,0.226780,0.867667,0.127976,0.954905,0.046408,1,250.078,255,39228.0,0.0,99.992,0.0,100.015,0.0,809.255,0.0,0.0,0.0,0.0,0.0,200.0,0.0,200.0,0.0,223.524,0.0,223.451,0.0,72.380,0.0,0.753,0.0,282.843,0.0,10003140.0,0.0,100.0,0.0,10003140.0,0.0,0.0,0.0,0.0,0.0,135.000,0.0,200.0,0.0,1.000,0.0,1.000,0.0,0.981,0.0
2,0.028367,0.057912,0.073998,0.080784,-0.133789,0.008512,0.534244,1,2,7.png,7,/content/Images3/,1,77,4.692,0,9.558442,1.427863,100.557675,58.022388,98.103338,59.633687,10.379961,1.091819,98.870130,58.006796,96.519481,59.576560,3.402597,0.564300,3.194805,0.559196,3.806623,0.403173,3.196818,0.352550,29.220831,46.845042,0.992974,0.032896,4.416805,0.320359,2.437403e+03,364.105032,100.0,0.0,2.437403e+03,364.105032,98.961039,57.996292,96.753247,59.526521,133.450130,23.616781,2.961039,0.375951,1.209857,0.220792,0.849429,0.129524,0.946805,0.050966,1,250.308,255,39264.0,0.0,99.987,0.0,100.001,0.0,808.326,0.0,0.0,0.0,0.0,0.0,200.0,0.0,200.0,0.0,223.626,0.0,223.554,0.0,5.766,0.0,0.755,0.0,282.843,0.0,10012320.0,0.0,100.0,0.0,10012320.0,0.0,0.0,0.0,200.0,0.0,45.000,0.0,200.0,0.0,1.000,0.0,1.000,0.0,0.982,0.0
3,0.036364,0.055723,0.088968,0.107719,-0.126763,0.011323,

In [17]:
# two dataframes 
# can eliminate this code b/c it is not nessesary. 

# Possibly df1 = feats_df_new
df1 = feats_df5[["noise","path","seed","dir","pred_class"]]
df2 = feats_df_new[["noise","path","seed","dir","pred_class"]]

# concat's predicted and manual classified dataframes (df_classes_classified + feats_df)
pred_and_manual_df = pd.concat([df1, df2], axis=0)

pred_and_manual_df.reset_index(inplace = True, drop = True)
feats_df_new.reset_index(inplace = True, drop = True)

pred_and_manual_df.tail(20)

KeyError: "['pred_class'] not in index"

In [ ]:
#add full path to each dataframe to allow matching based on unique path
# df with all params: feats_df_new
# df with predicted classes: pred_and_manual_df

fdf_full_path = []
for i in range(feats_df_new.shape[0]):
  fdf_full_path.append(feats_df_new["dir"][i] + feats_df_new["path"][i])


cdf_full_path = []
for j in range(pred_and_manual_df.shape[0]):
  cdf_full_path.append((pred_and_manual_df["dir"][j] + pred_and_manual_df["path"][j]))


feats_df_new["full_path"] = fdf_full_path
pred_and_manual_df["full_path"] = cdf_full_path

In [ ]:
## Indices 1 - ALl elements in feats_df_new that match full paths with rows in pred_and_manual_df.
## No need to run after done once.

indices_1 = []
indices_2 = []
errors = 0
counter = 0
for i in range(feats_df_new.shape[0]):
  #pull up path to search for
  path = pred_and_manual_df["full_path"][i]

  #find index within feats_df that matches path
  ind_df = feats_df_new[feats_df_new["full_path"] == path]
  #print(ind_df.head())
  #print("Ind_df.shape[0] ", ind_df.shape[0])
    
  if ind_df.shape[0] == 1:
    ind = ind_df.index[0]
    # print(ind_df.head())
#    #print("ind: " + str(ind) + " i: " + str(i))
    indices_1.append(ind)
    indices_2.append(i)
  else:
   # indices.append("error")
   # print("error")
    errors += 1


  if counter % 1000 == 0:
    print(counter)
  counter += 1

#print(ind_df.head())
#print(indices_2)
print("Complete.")

In [ ]:
# match dfs
# drops all values of feats_df_new that aren't in indices_1
# drops all values of pred_and_manual_df that aren't in indices_2

feats_df_new = feats_df_new.iloc[indices_1]
pred_and_manual_df = pred_and_manual_df.iloc[indices_2]
feats_df_new.reset_index(inplace = True, drop = True)
pred_and_manual_df.reset_index(inplace = True, drop = True)

In [ ]:
same = list(np.array(feats_df_new["full_path"]) == np.array(pred_and_manual_df["full_path"]))

print(same.count(True))
print(same.count(False))
print(len(same))

#print(same)

In [ ]:
feats_df_new["classifier_pred_class"] = pred_and_manual_df["pred_class"]

In [ ]:
#its a feats_df of constrained parameters
feats_df_new.head()

feats_df = feats_df_new

In [ ]:
print(feats_df.shape)

In [ ]:
print(feats_df.columns[13:111])

XGBOOST

In [8]:
#! pip install xgboost
#! pip install openpyxl
#! pip install tensorflow
#! pip install graphviz
#! pip install hyperopt

from sklearn.multioutput import MultiOutputRegressor
from sklearn.svm import SVR
import numpy as np
from sklearn.model_selection import RepeatedKFold
from numpy import absolute
from pandas import read_csv
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import RepeatedKFold
from xgboost import XGBRegressor
import openpyxl
from xgboost import cv
from PIL import Image
import pandas as pd
import os
import numpy as np
import matplotlib.pyplot as plt
import operator
# for loading/processing the images
import tensorflow
from tensorflow.keras.utils import load_img
from tensorflow.keras.utils import img_to_array
from keras.applications.vgg16 import preprocess_input

# models
from keras.applications.vgg16 import VGG16
from keras.models import Model

# clustering and dimension reduction
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
import pickle
from sklearn.ensemble import RandomForestRegressor
from sklearn import tree
import graphviz
from sklearn import metrics
from sklearn.metrics import r2_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.model_selection import GridSearchCV
import xgboost as xgb
from hyperopt import fmin, tpe, hp,STATUS_OK
from sklearn.model_selection import KFold, cross_val_score
from sklearn.model_selection import KFold
from sklearn.model_selection import cross_val_score

print("All dependencies present")

C:\Users\user.SR7872\AppData\Local\Programs\Python\Python312\Lib\site-packages\hyperopt\atpe.py:19: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


All dependencies present


In [ ]:
np.unique(feats_df["num_spots"].astype(float))

In [23]:
# scale data 
# uses Standard Scaler, range [-1, 1]

## X (independent variable) --> the feature values themselves
X = feats_df5[feats_df.columns[13:111]].astype(float)

## Y (dependent variables) --> the parameter values of Negan and RTO themselves
y = feats_df5[["Ua","Ui","Ga","Gi","Da","Di","Ba"]].astype(float)

#Scale data with standardscaler
scaling = StandardScaler()

# Use fit and transform method
scaling.fit(X)
X_scaled = scaling.transform(X)

# select 20 percent for testing
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42) #create training split



# df_train = feats_df.sample(frac = 0.80, random_state = 42)
# df_train.reset_index(inplace = True, drop = True)
# df_val_test = feats_df.drop(df_train.index)

# p_out = 1

# df_val_test = df_val_test.sample(frac = p_out, random_state = 42)
# df_val = df_val_test.sample(frac = 0.5, random_state = 42)
# df_test = df_val_test.drop(df_val.index)

# df_val.reset_index(inplace = True, drop = True)
# df_test.reset_index(inplace = True, drop = True)

In [24]:
for column in feats_df5.columns:
    print(column)

Ua
Ui
Ga
Gi
Ba
Da
Di
pattern
noise
path
seed
dir
feat_bool
num_spots
Mean
Median
Area_mean
Area_std
X_mean
X_std
Y_mean
Y_std
Perim._mean
Perim._std
BX_mean
BX_std
BY_mean
BY_std
Width_mean
Width_std
Height_mean
Height_std
Major_mean
Major_std
Minor_mean
Minor_std
Angle_mean
Angle_std
Circ._mean
Circ._std
Feret_mean
Feret_std
IntDen_mean
IntDen_std
%Area_mean
%Area_std
RawIntDen_mean
RawIntDen_std
FeretX_mean
FeretX_std
FeretY_mean
FeretY_std
FeretAngle_mean
FeretAngle_std
MinFeret_mean
MinFeret_std
AR_mean
AR_std
Round_mean
Round_std
Solidity_mean
Solidity_std
num_spots_inverted
Mean_inverted
Median_inverted
Area_inverted_mean
Area_inverted_std
X_inverted_mean
X_inverted_std
Y_inverted_mean
Y_inverted_std
Perim._inverted_mean
Perim._inverted_std
BX_inverted_mean
BX_inverted_std
BY_inverted_mean
BY_inverted_std
Width_inverted_mean
Width_inverted_std
Height_inverted_mean
Height_inverted_std
Major_inverted_mean
Major_inverted_std
Minor_inverted_mean
Minor_inverted_std
Angle_inverted_mean

In [25]:
y_test

,Ua,Ui,Ga,Gi,Da,Di,Ba
6706,0.046849,0.013064,0.115779,0.033348,0.011170,0.616537,-0.118356
19387,0.022846,0.156399,0.118046,0.069050,0.009939,1.052442,-0.193037
14847,0.045896,0.199486,0.061461,0.078881,0.009406,0.433193,-0.097045
24306,0.024930,0.051862,0.094625,0.024345,0.010236,0.498789,-0.098016
7172,0.030000,0.070000,0.080000,0.100000,0.010000,0.827444,-0.120000
...,...,...,...,...,...,...,...
12968,0.014529,0.138111,0.112518,0.063431,0.010815,1.113066,-0.100108
1891,0.009662,0.195854,0.053540,0.101883,0.011343,1.117866,-0.109434
23541,0.045216,0.168554,0.076138,0.056090,0.011093,0.641815,-0.057450
16741,0.020215,0.142634,0.059699,0.142708,0.010567,0.614994,-0.108160


In [26]:
model = xgb.XGBRegressor(n_estimators=1000, max_depth=10, eta=0.1, subsample=0.7, colsample_bytree=0.8, num_boost_round=50, objective= "reg:squarederror", device = "cuda")
model.fit(X_train, y_train)

C:\Users\user.SR7872\AppData\Local\Programs\Python\Python312\Lib\site-packages\xgboost\training.py:183: UserWarning: [14:47:12] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\context.cc:49: No visible GPU is found, setting device to CPU.
  bst.update(dtrain, iteration=i, fobj=obj)
C:\Users\user.SR7872\AppData\Local\Programs\Python\Python312\Lib\site-packages\xgboost\training.py:183: UserWarning: [14:47:12] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\context.cc:203: XGBoost is not compiled with CUDA support.
  bst.update(dtrain, iteration=i, fobj=obj)
C:\Users\user.SR7872\AppData\Local\Programs\Python\Python312\Lib\site-packages\xgboost\training.py:183: UserWarning: [14:47:12] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "num_boost_round" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=0.8, device='cuda', early_stopping_rounds=None,
             enable_categorical=False, eta=0.1, eval_metric=None,
             feature_types=None, feature_weights=None, gamma=None,
             grow_policy=None, importance_type=None,
             interaction_constraints=None, learning_rate=None, max_bin=None,
             max_cat_threshold=None, max_cat_to_onehot=None,
             max_delta_step=None, max_depth=10, max_leaves=None,
             min_child_weight=None, missing=nan, monotone_constraints=None,
             multi_strategy=None, n_estimators=1000, n_jobs=None, ...)

In [38]:
## Save XGBoost Model to File

model.save_model(file_out)

C:\Users\user.SR7872\AppData\Local\Programs\Python\Python312\Lib\site-packages\xgboost\sklearn.py:1028: UserWarning: [15:04:33] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\c_api\c_api.cc:1427: Saving model in the UBJSON format as default.  You can use file extension: `json`, `ubj` or `deprecated` to choose between formats.
  self.get_booster().save_model(fname)


In [41]:
loaded_model = xgb.XGBRegressor()

loaded_model.load_model(file_out)

#loaded_model.fit(X_train, y_train)

# make predictions
y_pred = loaded_model.predict(X_test)

pd.DataFrame(data=y_pred) 

# Columns - parameters 1-7
# Rows - Subsample of rows

print("Model loaded")

Model loaded


In [42]:
# GOAL: To determine the error rate in predicting the value of continuous, parameter data

# Converts test Y values to numpy array
true_vals = np.array(y_test)

# Outputs amount of true values
print(true_vals.shape)

# Outputs amount of predicted values
print(y_pred.shape)

# Sets counters for correct, incorrect, errors numpy array
correct = 0
incorrect = 0
p_errs = np.zeros(7)

# loops through each row in the true_vals array
for i in range(true_vals.shape[0]):
  # predicted value = predicted value from loop
  pred = y_pred[i]
  # true value = true value from loop
  true_val = true_vals[i]
  # Adds to error: absolute percent difference between true and predicted values
  p_errs += (np.abs((pred-true_val)/true_val))

# Outputs error percentages
print((p_errs/true_vals.shape[0])*100)

(5503, 7)
(5503, 7)
[33.01755118 58.23593786 26.19850456 41.53628171  8.9420851  38.46586172
 33.90088487]


In [35]:
# make a dataframe with the names and feat_dfs and inverse_feat_dfs:
real_imj_df = {"path":[],"feats_df":[],"inverse_feats_df":[]}

reg_paths = os.listdir("./content/ImageJ_data/Inverse_resized/")

for path in reg_paths:
  if path != ".ipynb_checkpoints":
    real_imj_df["path"].append(path)
    real_imj_df["feats_df"].append(pd.read_csv("./content/ImageJ_data/Regular_resized/" + path))
    real_imj_df["inverse_feats_df"].append(pd.read_csv("./content/ImageJ_data/Inverse_resized/" + path))

real_df = pd.DataFrame(real_imj_df)

In [37]:
real_df

,path,feats_df,inverse_feats_df
0,bHLH2_OE_(green)_(0-141_thresh.tif.csv,Label A...,Label Are...
1,bHLH2_RNAi_(green)_(0-170).tif.csv,Label Area M...,Label Area M...
2,F2_260_green)_(0-105).tif.csv,Label Area M...,Label Area Mea...
3,F2_A 8_(green)_(0-100).tif.csv,Label Area M...,Label Area Me...
4,LF10_11-20-23_(green)_(0-152).tif.csv,Label ...,Label Area...
5,MLC_F1_11-20-23_(green)_(0-105).tif.csv,Label ...,Label Ar...
6,mpar_rto(c-c)_(green)_(0-137).tif.csv,Label Ar...,Label Area...
7,mpar_rto_crispr_(green)_(0-155).tif.csv,Label Ar...,Label Ar...
8,mpar_wt_(green)_(0-170).tif.csv,Label Area ...,Label Area M...
9,NEGAN_crispr_(green)_(0-150).tif.csv,Label Area ...,Label Area ...


In [38]:
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)
pd.set_option('display.max_colwidth', None) 
real_df.shape

columns0 = ['feats_df', 'inverse_feats_df']

real_df2 = real_df.drop(columns=columns0)

real_df2

#real_df.head(14)

,path
0,bHLH2_OE_(green)_(0-141_thresh.tif.csv
1,bHLH2_RNAi_(green)_(0-170).tif.csv
2,F2_260_green)_(0-105).tif.csv
3,F2_A 8_(green)_(0-100).tif.csv
4,LF10_11-20-23_(green)_(0-152).tif.csv
5,MLC_F1_11-20-23_(green)_(0-105).tif.csv
6,mpar_rto(c-c)_(green)_(0-137).tif.csv
7,mpar_rto_crispr_(green)_(0-155).tif.csv
8,mpar_wt_(green)_(0-170).tif.csv
9,NEGAN_crispr_(green)_(0-150).tif.csv


In [39]:
# Add regular features

# first list is mean, second list is std except for mean and median
feats = {'Mean':[],'Median':[],'Area':[[],[]], 'X':[[],[]], 'Y':[[],[]], 'Perim.':[[],[]], 'BX':[[],[]],
         'BY':[[],[]], 'Width':[[],[]], 'Height':[[],[]], 'Major':[[],[]], 'Minor':[[],[]],
       'Angle':[[],[]], 'Circ.':[[],[]], 'Feret':[[],[]], 'IntDen':[[],[]], '%Area':[[],[]],
       'RawIntDen':[[],[]], 'FeretX':[[],[]], 'FeretY':[[],[]], 'FeretAngle':[[],[]], 'MinFeret':[[],[]], 'AR':[[],[]],
       'Round':[[],[]], 'Solidity':[[],[]]}

# feats = {'Mean_inverted':[],'Median_inverted':[],'Area':[[],[]], 'X':[[],[]], 'Y':[[],[]], 'Perim.':[[],[]], 'BX':[[],[]],
#          'BY':[[],[]], 'Width':[[],[]], 'Height':[[],[]], 'Major':[[],[]], 'Minor':[[],[]],
#        'Angle':[[],[]], 'Circ.':[[],[]], 'Feret':[[],[]], 'IntDen':[[],[]], '%Area':[[],[]],
#        'RawIntDen':[[],[]], 'FeretX':[[],[]], 'FeretY':[[],[]], 'FeretAngle':[[],[]], 'MinFeret':[[],[]], 'AR':[[],[]],
#        'Round':[[],[]], 'Solidity':[[],[]]}


counter = 0

# iterate through all sims
for n in range(real_df.shape[0]): # grab the feats dataframe for this sim
  fdf = real_df["feats_df"][n] #iterate through all feats within the feats dictionary

  for i in range(len(feats.keys())): # select the current feat
    feat = list(feats.keys())[i]

    # if feat in ['Mean_inverted','Median_inverted']: # check for feats that should be added from the overall measurement (first row)
    if feat in ['Mean','Median']: # check for feats that should be added from the overall measurement (first row)
      # feats[feat].append(fdf[feat[0:-9]][0]) #inverse
      feats[feat].append(fdf[feat[:]][0])
      # feats[feat].append(fdf[feat][0])

    else:
      if fdf.shape[0] <= 1: # check if the fdf is only one measurement (empty result)
        feats[feat][0].append(np.mean(np.array(fdf[feat][0]))) # mean of sole measurement
        feats[feat][1].append(np.std(np.array(fdf[feat][0]))) #std of sole measurement (0)

      else:
        feats[feat][0].append(np.mean(np.array(fdf[feat][1:]))) # add mean for each of the rest of the feats using the remaining rows
        feats[feat][1].append(np.std(np.array(fdf[feat][1:]))) # add std for each of the rest of the feats using the remaining rows

  if counter % 1000 == 0:
    print(counter)
  counter += 1

print("Counting done")

0
Counting done


In [40]:
num_spots = []

for i in range(real_df.shape[0]):
  fdf = real_df["feats_df"][i] #iterate through all feats within the feats dictionary
  if fdf.shape[0] <= 1:
    num_spots.append(0)
  else:
    num_spots.append(fdf.shape[0]-1)

# feats_df["num_spots_inverted"] = num_spots
real_df["num_spots"] = num_spots

In [41]:
for key in list(feats.keys()):
  # if key in ['Mean_inverted','Median_inverted']:
  if key in ['Mean','Median']:
    real_df[key] = feats[key]
  else:
    # feats_df[key + "_inverted_mean"] = feats[key][0]
    # feats_df[key + "_inverted_std"] = feats[key][1]
    real_df[key + "_mean"] = feats[key][0]
    real_df[key + "_std"] = feats[key][1]

In [42]:
# Add regular features

# first list is mean, second list is std except for mean and median
# feats = {'Mean':[],'Median':[],'Area':[[],[]], 'X':[[],[]], 'Y':[[],[]], 'Perim.':[[],[]], 'BX':[[],[]],
#          'BY':[[],[]], 'Width':[[],[]], 'Height':[[],[]], 'Major':[[],[]], 'Minor':[[],[]],
#        'Angle':[[],[]], 'Circ.':[[],[]], 'Feret':[[],[]], 'IntDen':[[],[]], '%Area':[[],[]],
#        'RawIntDen':[[],[]], 'FeretX':[[],[]], 'FeretY':[[],[]], 'FeretAngle':[[],[]], 'MinFeret':[[],[]], 'AR':[[],[]],
#        'Round':[[],[]], 'Solidity':[[],[]]}

feats = {'Mean_inverted':[],'Median_inverted':[],'Area':[[],[]], 'X':[[],[]], 'Y':[[],[]], 'Perim.':[[],[]], 'BX':[[],[]],
         'BY':[[],[]], 'Width':[[],[]], 'Height':[[],[]], 'Major':[[],[]], 'Minor':[[],[]],
       'Angle':[[],[]], 'Circ.':[[],[]], 'Feret':[[],[]], 'IntDen':[[],[]], '%Area':[[],[]],
       'RawIntDen':[[],[]], 'FeretX':[[],[]], 'FeretY':[[],[]], 'FeretAngle':[[],[]], 'MinFeret':[[],[]], 'AR':[[],[]],
       'Round':[[],[]], 'Solidity':[[],[]]}


counter = 0

# iterate through all sims
for n in range(real_df.shape[0]): # grab the feats dataframe for this sim
  fdf = real_df["inverse_feats_df"][n] #iterate through all feats within the feats dictionary

  for i in range(len(feats.keys())): # select the current feat
    feat = list(feats.keys())[i]

    if feat in ['Mean_inverted','Median_inverted']: # check for feats that should be added from the overall measurement (first row)
    # if feat in ['Mean','Median']: # check for feats that should be added from the overall measurement (first row)
      feats[feat].append(fdf[feat[0:-9]][0]) #inverse
      # feats[feat].append(fdf[feat[:]][0])

    else:
      if fdf.shape[0] <= 1: # check if the fdf is only one measurement (empty result)
        feats[feat][0].append(np.mean(np.array(fdf[feat][0]))) # mean of sole measurement
        feats[feat][1].append(np.std(np.array(fdf[feat][0]))) #std of sole measurement (0)

      else:
        feats[feat][0].append(np.mean(np.array(fdf[feat][1:]))) # add mean for each of the rest of the feats using the remaining rows
        feats[feat][1].append(np.std(np.array(fdf[feat][1:]))) # add std for each of the rest of the feats using the remaining rows

  if counter % 1000 == 0:
    print(counter)
  counter += 1

print("Counter complete")

0
Counter complete


In [43]:
num_spots = []

for i in range(real_df.shape[0]):
  fdf = real_df["inverse_feats_df"][i] #iterate through all feats within the feats dictionary
  if fdf.shape[0] <= 1:
    num_spots.append(0)
  else:
    num_spots.append(fdf.shape[0]-1)

real_df["num_spots_inverted"] = num_spots
# real_df["num_spots"] = num_spots

In [44]:
for key in list(feats.keys()):
  if key in ['Mean_inverted','Median_inverted']:
  # if key in ['Mean','Median']:
    real_df[key] = feats[key]
  else:
    real_df[key + "_inverted_mean"] = feats[key][0]
    real_df[key + "_inverted_std"] = feats[key][1]
    # real_df[key + "_mean"] = feats[key][0]
    # real_df[key + "_std"] = feats[key][1]

In [45]:
print(real_df.columns[3:101])

Index(['num_spots', 'Mean', 'Median', 'Area_mean', 'Area_std', 'X_mean', 'X_std', 'Y_mean', 'Y_std', 'Perim._mean', 'Perim._std', 'BX_mean', 'BX_std', 'BY_mean', 'BY_std', 'Width_mean', 'Width_std', 'Height_mean', 'Height_std', 'Major_mean', 'Major_std', 'Minor_mean', 'Minor_std', 'Angle_mean', 'Angle_std', 'Circ._mean', 'Circ._std', 'Feret_mean', 'Feret_std', 'IntDen_mean', 'IntDen_std', '%Area_mean', '%Area_std', 'RawIntDen_mean', 'RawIntDen_std', 'FeretX_mean', 'FeretX_std', 'FeretY_mean', 'FeretY_std', 'FeretAngle_mean', 'FeretAngle_std', 'MinFeret_mean', 'MinFeret_std', 'AR_mean', 'AR_std', 'Round_mean', 'Round_std', 'Solidity_mean', 'Solidity_std', 'num_spots_inverted', 'Mean_inverted', 'Median_inverted', 'Area_inverted_mean', 'Area_inverted_std', 'X_inverted_mean', 'X_inverted_std', 'Y_inverted_mean', 'Y_inverted_std', 'Perim._inverted_mean', 'Perim._inverted_std', 'BX_inverted_mean', 'BX_inverted_std', 'BY_inverted_mean', 'BY_inverted_std', 'Width_inverted_mean',
       'Width_

In [30]:
real_feats.head()
#real_df.head()

NameError: name 'real_feats' is not defined

In [18]:
new_df = pd.read_pickle("./data/Images_Classified_np126.pkl")

new_feats = new_df[new_df.columns[13:101]]

new_feats.head()

,num_spots,Mean,Median,Area_mean,Area_std,X_mean,X_std,Y_mean,Y_std,Perim._mean,...,IntDen_inverted_mean,IntDen_inverted_std,%Area_inverted_mean,%Area_inverted_std,RawIntDen_inverted_mean,RawIntDen_inverted_std,FeretX_inverted_mean,FeretX_inverted_std,FeretY_inverted_mean,FeretY_inverted_std
0,70,5.489,0,12.300000,2.548669,99.395943,58.506181,97.530229,58.913237,12.282143,...,9980445.0,0.0,100.0,0.0,9980445.0,0.0,0.0,0.0,200.0,0.0
1,63,4.921,0,12.253968,2.569468,101.800571,59.297600,99.451270,58.953924,11.663270,...,10003140.0,0.0,100.0,0.0,10003140.0,0.0,0.0,0.0,0.0,0.0
2,77,4.692,0,9.558442,1.427863,100.557675,58.022388,98.103338,59.633687,10.379961,...,10012320.0,0.0,100.0,0.0,10012320.0,0.0,0.0,0.0,200.0,0.0
3,73,7.931,0,17.041096,3.365572,103.192973,59.580318,99.320973,58.524761,13.973644,...,9882780.0,0.0,100.0,0.0,9882780.0,0.0,0.0,0.0,0.0,0.0
4,75,8.154,0,17.053333,3.905187,104.204653,61.052503,99.000587,59.296666,14.229160,...,9873855.0,0.0,100.0,0.0,9873855.0,0.0,0.0,0.0,0.0,0.0


In [29]:
real_df

NameError: name 'real_df' is not defined

In [43]:
### TODO: Re-run this and then compare with the above results. ALso, purge unnesesary cells that cause problems.

new_df = pd.read_pickle("./data/real_df_xgboost.pkl")

new_feats = new_df[new_df.columns[13:111]]

new_feats.head()


y_pred = None

loaded_model = xgb.XGBRegressor()

loaded_model.load_model(file_out)

# # select features of real images
# #real_feats = real_df[real_df.columns[3:101]]

# #scale data
scaling=StandardScaler()

# # Use fit and transform method
scaling.fit(new_feats)
new_feats_scaled = scaling.transform(real_feats)

# # predict params of images from real feats
# y_pred = loaded_model.predict(new_feats_scaled)

# print(y_pred[2])

# new_df["pred_params"] = list(y_pred)

In [48]:
new_df

,Ua,Ui,Ga,Gi,Ba,Da,Di,pattern,noise,path,...,MinFeret_inverted_std,AR_inverted_mean,AR_inverted_std,Round_inverted_mean,Round_inverted_std,Solidity_inverted_mean,Solidity_inverted_std,full_path,classifier_pred_class,pred_params
0,0.025122,0.063822,0.071957,0.106327,-0.113131,0.009186,0.611871,1,2,1.png,...,0.0,1.001,0.0,0.999,0.0,0.979,0.0,/content/Images3/1.png,2,"[0.035205893, 0.10500387, 0.08964063, 0.105052..."
1,0.034475,0.059956,0.080798,0.10037,-0.132577,0.008852,0.854084,1,2,3.png,...,0.0,1.000,0.0,1.000,0.0,0.981,0.0,/content/Images3/3.png,2,"[0.044258676, 0.08403519, 0.08878907, 0.109108..."
2,0.028367,0.057912,0.073998,0.080784,-0.133789,0.008512,0.534244,1,2,7.png,...,0.0,1.000,0.0,1.000,0.0,0.982,0.0,/content/Images3/7.png,2,"[0.041021343, 0.10885004, 0.08606631, 0.110363..."
3,0.035847,0.078762,0.07823,0.101485,-0.115752,0.010542,0.933894,1,2,12.png,...,0.0,1.000,0.0,1.000,0.0,0.969,0.0,/content/Images3/12.png,2,"[0.045711964, 0.120173216, 0.09000798, 0.10781..."
4,0.029837,0.067296,0.079054,0.095099,-0.132865,0.009721,0.822421,1,2,13.png,...,0.0,1.000,0.0,1.000,0.0,0.968,0.0,/content/Images3/13.png,2,"[0.042650323, 0.12191723, 0.08885314, 0.119624..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
27510,0.034675,0.083466,0.168377,0.037123,-0.158271,0.00931,1.086042,0,4,noise 4 20872.0.png,...,0.0,1.000,0.0,1.000,0.0,1.000,0.0,/content/Images1/noise 4 20872.0.png,1,"[0.02365914, 0.11698489, 0.12690276, 0.0905121..."
27511,0.04988,0.013225,0.060507,0.11463,-0.154332,0.010548,0.255686,0,4,noise 4 22801.0.png,...,0.0,1.000,0.0,1.000,0.0,1.000,0.0,/content/Images1/noise 4 22801.0.png,0,"[0.033132907, 0.08536519, 0.071761526, 0.11440..."
27512,0.045854,0.209092,0.09992,0.0442,-0.116251,0.010908,1.397959,0,5,noise 5 21620.0.png,...,0.0,1.000,0.0,1.000,0.0,1.000,0.0,/content/Images1/noise 5 21620.0.png,1,"[0.02365914, 0.11698489, 0.12690276, 0.0905121..."
27513,0.047659,0.179187,0.122823,0.048745,-0.080442,0.011393,0.468993,0,3,noise 3 21522.0.png,...,0.0,1.000,0.0,1.000,0.0,1.000,0.0,/content/Images1/noise 3 21522.0.png,1,"[0.02365914, 0.11698489, 0.12690276, 0.0905121..."


In [49]:
real_df["pred_params"][0]

array([ 0.02614826,  0.06258593,  0.12087097,  0.08383078,  0.01085906,
        0.85632795, -0.09742121], dtype=float32)

In [50]:
## TODO: Grab dataframe at the top (w/ all the features), drop all unnessesary columns (diff. from real_df, drop parameters)
## choose first five images and run it and see how close they are. 

for i in range(real_df.shape[0]):
  print(real_df["path"][i])
  print(real_df["pred_params"][i])

bHLH2_OE_(green)_(0-141_thresh.tif.csv
[ 0.02614826  0.06258593  0.12087097  0.08383078  0.01085906  0.85632795
 -0.09742121]
bHLH2_RNAi_(green)_(0-170).tif.csv
[ 0.02351721  0.05714148  0.07845976  0.1292734   0.01027878  0.76815784
 -0.08877929]
F2_260_green)_(0-105).tif.csv
[ 0.02331874  0.06008431  0.12086973  0.11648673  0.00954962  0.70141786
 -0.0975161 ]
F2_A 8_(green)_(0-100).tif.csv
[ 0.02924089  0.06581408  0.1192591   0.10536818  0.00979359  0.6629838
 -0.06067621]
LF10_11-20-23_(green)_(0-152).tif.csv
[ 0.0221226   0.10830527  0.12630259  0.11860634  0.0105213   0.6667085
 -0.10096822]
MLC_F1_11-20-23_(green)_(0-105).tif.csv
[ 0.02128733  0.12389465  0.09693768  0.12160463  0.01039962  0.81738204
 -0.09693675]
mpar_rto(c-c)_(green)_(0-137).tif.csv
[ 0.02338564  0.11086314  0.10433803  0.11101844  0.01052789  0.75643164
 -0.08920884]
mpar_rto_crispr_(green)_(0-155).tif.csv
[ 0.01959674  0.11570201  0.12073527  0.10527825  0.00973131  0.9733093
 -0.08260424]
mpar_wt_(green)_